In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### Load / File / Generate

This cell documents the traffic scene object detection experiment. For the full original methodology, mathematical derivations, and commentary, see the archived notebook in `archive/original_notebooks/`.


In [2]:
# Note on Load. Refer to the archived source for the detailed original comment.
classes_path = "/kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/labels/classes.txt"
with open(classes_path, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]

nc = len(class_names)
names = {i: name for i, name in enumerate(class_names)}

print(f"检测到 {nc} 个类别:")
for i, name in names.items():
    print(f"  {i}: {name}")

# 写入 d2-city.yaml
yaml_content = f"""
# D2-City dataset configuration
path: /kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset
train: /kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/images/train
val: //kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/images/val

nc: {nc}
names: {names}
"""

with open("/kaggle/working/d2-city.yaml", "w") as f:
    f.write(yaml_content)

print("已生成 /kaggle/working/d2-city.yaml")

检测到 12 个类别:
  0: car
  1: van
  2: bus
  3: truck
  4: person
  5: bicycle
  6: motorcycle
  7: open-tricycle
  8: closed-tricycle
  9: forklift
  10: large-block
  11: small-block
已生成 /kaggle/working/d2-city.yaml


In [3]:
%%writefile train.py
"""
Part I: YOLO26 Training Script
"""

import os
from ultralytics import YOLO

def main():

    # Note on Model, Load. Refer to the archived source for the detailed original comment.
    model = YOLO("yolo26n.yaml")

    # Note on Training. Refer to the archived source for the detailed original comment.
    results = model.train(
        data="/kaggle/working/d2-city.yaml",  # 数据集配置文件
        save_dir="./runs",
        pretrained=False,  # 不加载预训练模型
        cache=False,   # 关闭缓存
        # Note on Hyperparameters, Parameters. Refer to the archived source for the detailed original comment.
        amp=False,  # 是否开启 Automatic Mixed Precision (AMP)
        epochs=100,  # 训练轮数 (默认: 100)
    )

    print("Training complete.")
    if results and hasattr(results, 'save_dir'):
        save_dir = results.save_dir
        print(f"\n训练结果保存在: {save_dir}")
        print("生成的文件:")
        for file in os.listdir(save_dir):
            print(f"  - {file}")

    # Note on Model, Validation, Evaluation. Refer to the archived source for the detailed original comment.
    metrics = model.val()

if __name__ == "__main__":
    main()


Writing train.py


In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.8 MB/s eta 0:00:00


In [5]:
!python train.py

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.103 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/d2-city.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h

In [6]:
from ultralytics import YOLO
import os
import random

# Note on Model, Training, Load. Refer to the archived source for the detailed original comment.
model = YOLO("/kaggle/working/runs/weights/best.pt")  # 或 last.pt

# Note on Testing, Image. Refer to the archived source for the detailed original comment.
test_img_dir = "/kaggle/input/datasets/xiaoyuwan/didi-detection/dataset/dataset_release/test"

# 随机选8个编号 (11000~11999)
indices = random.sample(range(11000, 12000), 8)
test_img_files = [os.path.join(test_img_dir, f"{idx}.jpg") for idx in indices]

# Note on Prediction. Refer to the archived source for the detailed original comment.
results = model(test_img_files)

# Note on Results. Refer to the archived source for the detailed original comment.
for file, result in zip(test_img_files, results):
    print(f"预测图像: {os.path.basename(file)}")
    result.show()


0: 640x640 6 cars, 7.8ms
1: 640x640 3 cars, 1 van, 7.8ms
2: 640x640 5 cars, 7.8ms
3: 640x640 6 cars, 7.8ms
4: 640x640 4 cars, 7.8ms
5: 640x640 3 cars, 1 van, 7.8ms
6: 640x640 8 cars, 3 vans, 1 bus, 1 truck, 7.8ms
7: 640x640 1 car, 7.8ms
Speed: 3.5ms preprocess, 7.8ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)
预测图像: 11804.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

预测图像: 11832.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

预测图像: 11076.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

... (6 outputs left out)

预测图像: 11341.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

预测图像: 11003.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>